# MLflow Example: Tracking Two Model Training Runs

In [1]:
! pip3 install scikit-learn


[notice] A new release of pip is available: 24.0 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# === Model Training and Experimentation ===

# Import necessary libraries
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.datasets import load_digits


In [3]:

# Load dataset
data = load_digits()
X_train, X_test, y_train, y_test = train_test_split(data.data, data.target, test_size=0.2, random_state=42)

# Define a function to train a model and return its accuracy
def train_model(n_estimators, max_depth):
    # Train a RandomForestClassifier
    model = RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth, random_state=42)
    model.fit(X_train, y_train)

    # Make predictions
    predictions = model.predict(X_test)
    accuracy = accuracy_score(y_test, predictions)

    print(f"Model with n_estimators={n_estimators}, max_depth={max_depth} achieved accuracy={accuracy:.4f}")
    return model, accuracy

In [4]:

# Train models with different hyperparameter configurations
print("Training Model 1...")
model_1, acc_1 = train_model(n_estimators=20, max_depth=5)

Training Model 1...
Model with n_estimators=20, max_depth=5 achieved accuracy=0.9389


In [5]:
print("\nTraining Model 2...")
model_2, acc_2 = train_model(n_estimators=100, max_depth=10)



Training Model 2...
Model with n_estimators=100, max_depth=10 achieved accuracy=0.9722


Now adding MLFlow

In [6]:
! pip3 install mlflow


[notice] A new release of pip is available: 24.0 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
import mlflow
import mlflow.sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.datasets import load_digits

C:\Users\thoma\Exo_MLOps\Ex7\mlflow-with-daghub\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
!pip3 install dagshub


[notice] A new release of pip is available: 24.0 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [9]:
# get this code from dagshub
import dagshub
dagshub.init(repo_owner='Redtom67', repo_name='mlops-ex7', mlflow=True)

Accessing as Redtom67

Initialized MLflow to track repo "Redtom67/mlops-ex7"

Repository Redtom67/mlops-ex7 initialized!

In [10]:
data = load_digits()
X_train, X_test, y_train, y_test = train_test_split(data.data, data.target, test_size=0.2, random_state=42)

In [11]:
# Set MLflow experiment name. This will get created if it doesn't exist
experiment_name = "RandomForest_V1.0.1"
mlflow.set_experiment(experiment_name)


<Experiment: artifact_location='mlflow-artifacts:/2b0a5d1eb8714fdc96534d30a4cd5e65', creation_time=1794841368934, effective_trace_archival_retention=None, experiment_id='0', last_update_time=1794841368934, lifecycle_stage='active', name='RandomForest_V1.0.1', tags={}, trace_location=None, workspace='default'>

In [14]:
# Name of the model in the MLflow Model Registry (used later by the FastAPI/Docker image)
REGISTERED_MODEL_NAME = "digits-rf"

# Define a function to train a model, log parameters and metrics to MLflow
def train_and_log_model(n_estimators, max_depth, run_name):
    with mlflow.start_run(run_name=run_name): # <--
        # Train a RandomForestClassifier
        model = RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth, random_state=42)
        model.fit(X_train, y_train)

        # Make predictions
        predictions = model.predict(X_test)
        accuracy = accuracy_score(y_test, predictions)

        # Log parameters, metrics, and the model itself
        mlflow.log_param("n_estimators", n_estimators)
        mlflow.log_param("max_depth", max_depth)
        mlflow.log_metric("accuracy", accuracy)
        mlflow.sklearn.log_model(
            model,
            name="model",
            registered_model_name=REGISTERED_MODEL_NAME,  # registers a new version in the MLflow Model Registry
            # MLflow 3 saves sklearn models with skops, which refuses tree structures unless we
            # explicitly trust them. We trained this model ourselves, so trusting it is safe.
            skops_trusted_types=["sklearn.tree._tree.Tree"],
        )

        print(f"Logged RandomForest model with n_estimators={n_estimators}, max_depth={max_depth}, accuracy={accuracy:.4f}")


In [15]:
# Train and log Model 1
print("Training and Logging Model 1...")
train_and_log_model(n_estimators=20, max_depth=5, run_name="random_forest_model_run_1")

Training and Logging Model 1...


Successfully registered model 'digits-rf'.
2026/11/16 16:09:46 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: digits-rf, version 1
Created version '1' of model 'digits-rf'.


Logged RandomForest model with n_estimators=20, max_depth=5, accuracy=0.9389
🏃 View run random_forest_model_run_1 at: https://dagshub.com/Redtom67/mlops-ex7.mlflow/#/experiments/0/runs/fdb2cbaa6f2241bab394429816f7af7a
🧪 View experiment at: https://dagshub.com/Redtom67/mlops-ex7.mlflow/#/experiments/0


In [16]:
# Train and log Model 2
print("\nTraining and Logging Model 2...")
train_and_log_model(n_estimators=100, max_depth=10, run_name="random_forest_model_run_V1.0.2")


Training and Logging Model 2...


Registered model 'digits-rf' already exists. Creating a new version of this model...
2026/11/16 16:12:03 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: digits-rf, version 2
Created version '2' of model 'digits-rf'.


Logged RandomForest model with n_estimators=100, max_depth=10, accuracy=0.9722
🏃 View run random_forest_model_run_V1.0.2 at: https://dagshub.com/Redtom67/mlops-ex7.mlflow/#/experiments/0/runs/b1f304d80665483da6375ddebdd8f4b9
🧪 View experiment at: https://dagshub.com/Redtom67/mlops-ex7.mlflow/#/experiments/0


In [17]:
# Instructions to visualize results
print("\nTo view the results, run the following command in your terminal:")
print("mlflow ui")
print("Then navigate to http://127.0.0.1:5000 to explore the experiment results.")


To view the results, run the following command in your terminal:
mlflow ui
Then navigate to http://127.0.0.1:5000 to explore the experiment results.
